In [34]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/integrated_data.csv', parse_dates=['date'])
features = {}

coins = ['BTC', 'ETH', 'BNB', 'ADA', 'SOL', 'DOGE']

# Price features for each coin
for col in coins + ['close']:
    features[f'{col}_ret_1d'] = df[col].pct_change()
    features[f'{col}_ret_7d'] = df[col].pct_change(7)
    features[f'{col}_ret_14d'] = df[col].pct_change(14)
    features[f'{col}_ret_30d'] = df[col].pct_change(30)
    features[f'{col}_log_ret'] = np.log(df[col] / df[col].shift(1).replace(0, np.nan))
    features[f'{col}_ma_7'] = df[col].rolling(7).mean()
    features[f'{col}_ma_14'] = df[col].rolling(14).mean()
    features[f'{col}_ma_30'] = df[col].rolling(30).mean()
    features[f'{col}_ma_60'] = df[col].rolling(60).mean()
    features[f'{col}_ma_ratio_7_30'] = features[f'{col}_ma_7'] / features[f'{col}_ma_30']
    features[f'{col}_ma_ratio_30_60'] = features[f'{col}_ma_30'] / features[f'{col}_ma_60']
    features[f'{col}_std_7'] = df[col].rolling(7).std()
    features[f'{col}_std_14'] = df[col].rolling(14).std()
    features[f'{col}_std_30'] = df[col].rolling(30).std()
    features[f'{col}_zscore_30'] = (df[col] - features[f'{col}_ma_30']) / features[f'{col}_std_30']
    features[f'{col}_zscore_60'] = (df[col] - features[f'{col}_ma_60']) / df[col].rolling(60).std()

# Volatility
features['daily_range'] = (df['high'] - df['low']) / df['close']
features['daily_range_pct'] = (df['high'] - df['low']) / df['open']
features['intraday_volatility'] = features['daily_range'].rolling(7).mean()
features['atr_7'] = features['daily_range'].rolling(7).mean()
features['atr_14'] = features['daily_range'].rolling(14).mean()
features['atr_30'] = features['daily_range'].rolling(30).mean()
features['volatility_trend'] = features['atr_7'] / features['atr_30']
features['realized_vol_7'] = features['close_log_ret'].rolling(7).std() * np.sqrt(365)
features['realized_vol_30'] = features['close_log_ret'].rolling(30).std() * np.sqrt(365)

# Volume
features['volume_ma_7'] = df['volume'].rolling(7).mean()
features['volume_ma_14'] = df['volume'].rolling(14).mean()
features['volume_ma_30'] = df['volume'].rolling(30).mean()
features['volume_ratio_7'] = df['volume'] / features['volume_ma_7']
features['volume_ratio_30'] = df['volume'] / features['volume_ma_30']
features['volume_change_7d'] = df['volume'].pct_change(7)
features['volume_std_30'] = df['volume'].rolling(30).std()
features['volume_zscore'] = (df['volume'] - features['volume_ma_30']) / features['volume_std_30']
features['price_volume_corr_30'] = df['close'].rolling(30).corr(df['volume'])

# RSI
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = delta.clip(upper=0).abs()
features['rsi_7'] = 100 - 100 / (1 + gain.rolling(7).mean() / loss.rolling(7).mean().replace(0, np.nan))
features['rsi_14'] = 100 - 100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean().replace(0, np.nan))
features['rsi_30'] = 100 - 100 / (1 + gain.rolling(30).mean() / loss.rolling(30).mean().replace(0, np.nan))

# Momentum
features['momentum_7'] = df['close'] - df['close'].shift(7)
features['momentum_14'] = df['close'] - df['close'].shift(14)
features['momentum_30'] = df['close'] - df['close'].shift(30)
features['roc_7'] = (df['close'] - df['close'].shift(7)) / df['close'].shift(7) * 100
features['roc_14'] = (df['close'] - df['close'].shift(14)) / df['close'].shift(14) * 100
features['roc_30'] = (df['close'] - df['close'].shift(30)) / df['close'].shift(30) * 100
features['price_position_30'] = (df['close'] - df['close'].rolling(30).min()) / (df['close'].rolling(30).max() - df['close'].rolling(30).min())
features['price_position_60'] = (df['close'] - df['close'].rolling(60).min()) / (df['close'].rolling(60).max() - df['close'].rolling(60).min())

# Sentiment
features['fear_greed'] = df['value']
features['fear_greed_ma_7'] = df['value'].rolling(7).mean()
features['fear_greed_ma_14'] = df['value'].rolling(14).mean()
features['fear_greed_ma_30'] = df['value'].rolling(30).mean()
features['fear_greed_std_30'] = df['value'].rolling(30).std()
features['fear_greed_zscore'] = (df['value'] - features['fear_greed_ma_30']) / features['fear_greed_std_30']
features['fear_greed_change_1d'] = df['value'].diff()
features['fear_greed_change_7d'] = df['value'].diff(7)
features['fear_greed_change_14d'] = df['value'].diff(14)
features['fear_greed_momentum'] = features['fear_greed_ma_7'] - features['fear_greed_ma_30']
features['extreme_greed'] = (df['value'] > 75).astype(int)
features['extreme_fear'] = (df['value'] < 25).astype(int)
features['greed_streak'] = (df['value'] > 60).astype(int).groupby((df['value'] <= 60).cumsum()).cumsum()
features['fear_streak'] = (df['value'] < 40).astype(int).groupby((df['value'] >= 40).cumsum()).cumsum()
features['sentiment_price_corr_30'] = df['value'].rolling(30).corr(features['close_ret_1d'])
features['sentiment_price_divergence'] = features['fear_greed_ma_7'] - features['close_zscore_30'] * 25 + 50

# Macro
features['fed_funds_change_1d'] = df['fed_funds'].diff()
features['fed_funds_change_7d'] = df['fed_funds'].diff(7)
features['fed_funds_change_30d'] = df['fed_funds'].diff(30)
features['fed_funds_ma_30'] = df['fed_funds'].rolling(30).mean()
features['treasury_10y_change_1d'] = df['treasury_10y'].diff()
features['treasury_10y_change_7d'] = df['treasury_10y'].diff(7)
features['treasury_10y_change_30d'] = df['treasury_10y'].diff(30)
features['treasury_10y_ma_30'] = df['treasury_10y'].rolling(30).mean()
features['yield_spread'] = df['treasury_10y'] - df['fed_funds']
features['yield_spread_change'] = features['yield_spread'].diff(7)
features['vix_ma_7'] = df['vix'].rolling(7).mean()
features['vix_ma_30'] = df['vix'].rolling(30).mean()
features['vix_std_30'] = df['vix'].rolling(30).std()
features['vix_zscore'] = (df['vix'] - features['vix_ma_30']) / features['vix_std_30']
features['vix_change_1d'] = df['vix'].diff()
features['vix_change_7d'] = df['vix'].diff(7)
features['vix_regime'] = (df['vix'] > 20).astype(int) + (df['vix'] > 30).astype(int)
features['m2_growth_30d'] = df['m2'].pct_change(30)
features['m2_growth_60d'] = df['m2'].pct_change(60)
features['m2_growth_90d'] = df['m2'].pct_change(90)
features['m2_ma_30'] = df['m2'].rolling(30).mean()
features['m2_acceleration'] = features['m2_growth_30d'] - features['m2_growth_30d'].shift(30)
features['cpi_change_30d'] = df['cpi'].diff(30)
features['cpi_ma_30'] = df['cpi'].rolling(30).mean()
features['real_rate'] = df['treasury_10y'] - df['cpi']
features['real_rate_change'] = features['real_rate'].diff(30)
features['btc_vix_corr_30'] = df['BTC'].rolling(30).corr(df['vix'])
features['btc_m2_corr_60'] = df['BTC'].rolling(60).corr(df['m2'])
features['risk_off_signal'] = ((df['vix'] > features['vix_ma_30']) & (df['treasury_10y'] < features['treasury_10y_ma_30'])).astype(int)

# Cross asset
features['btc_eth_corr_30'] = df['BTC'].rolling(30).corr(df['ETH'])
features['btc_eth_corr_60'] = df['BTC'].rolling(60).corr(df['ETH'])
features['btc_alt_corr_avg'] = (df['BTC'].rolling(30).corr(df['ETH']) + df['BTC'].rolling(30).corr(df['BNB']) + df['BTC'].rolling(30).corr(df['ADA']) + df['BTC'].rolling(30).corr(df['SOL']) + df['BTC'].rolling(30).corr(df['DOGE'])) / 5
features['eth_alt_corr_avg'] = (df['ETH'].rolling(30).corr(df['BNB']) + df['ETH'].rolling(30).corr(df['ADA']) + df['ETH'].rolling(30).corr(df['SOL']) + df['ETH'].rolling(30).corr(df['DOGE'])) / 4
total_price = df['BTC'] + df['ETH'] + df['BNB'] + df['ADA'] + df['SOL'] + df['DOGE']
features['btc_dominance_proxy'] = df['BTC'] / total_price
features['btc_dominance_change'] = features['btc_dominance_proxy'].diff(7)

# Lagged features
lag_cols = ['close_ret_1d', 'close_ret_7d', 'volume_ratio_30', 'fear_greed', 'rsi_14', 'btc_eth_corr_30']
for lag in [1, 3, 7, 14]:
    for col in lag_cols:
        features[f'{col}_lag_{lag}'] = features[col].shift(lag)

features['day_of_week'] = df['date'].dt.dayofweek
features['month'] = df['date'].dt.month
features['quarter'] = df['date'].dt.quarter
features['year'] = df['date'].dt.year
features['is_month_start'] = df['date'].dt.is_month_start.astype(int)
features['is_month_end'] = df['date'].dt.is_month_end.astype(int)

base_cols = ['date', 'symbol', 'open', 'high', 'low', 'close', 'volume'] + coins + ['fed_funds', 'treasury_10y', 'vix', 'm2', 'cpi', 'value', 'value_classification']
feature_df = pd.concat([df[base_cols], pd.DataFrame(features)], axis=1)

feature_df = feature_df.iloc[90:].reset_index(drop=True)

# NA handling - median imputation for numeric features
num_cols = feature_df.select_dtypes(include=[np.number]).columns
feature_df[num_cols] = feature_df[num_cols].fillna(feature_df[num_cols].median())

# =============================================================================

feature_df.to_csv('data/df_all_v1.csv', index=False)

print(f"Shape: {feature_df.shape}")
print(f"Date range: {feature_df['date'].min()} to {feature_df['date'].max()}")
print(f"NaNs remaining: {feature_df.select_dtypes(include=[np.number]).isnull().sum().sum()}")

Shape: (2348, 242)
Date range: 2018-07-30 00:00:00 to 2025-01-01 00:00:00
NaNs remaining: 0


In [35]:
df.head(10)

,date,open,high,low,close,volume,symbol,BTC,ETH,BNB,ADA,SOL,DOGE,fed_funds,treasury_10y,vix,m2,cpi,value,value_classification
0,2018-05-01,0.34145,0.36034,0.31870,0.35504,1.203687e+08,ADA,8.0,3.0,9.0,33.0,52.0,61.0,1.7,2.97,15.49,14077.4,250.792,56.0,Greed
1,2018-05-02,0.35498,0.37586,0.34500,0.37199,6.545769e+07,ADA,8.0,3.0,9.0,30.0,51.0,58.0,1.7,2.97,15.97,14077.4,250.792,52.0,Neutral
2,2018-05-03,0.37199,0.38850,0.36402,0.36716,8.832641e+07,ADA,9.0,4.0,8.0,31.0,53.0,60.0,1.7,2.94,15.90,14077.4,250.792,55.0,Greed
3,2018-05-04,0.36683,0.36897,0.34650,0.35800,5.606927e+07,ADA,9.0,3.0,8.0,30.0,55.0,61.0,1.7,2.95,14.77,14077.4,250.792,56.0,Greed
4,2018-05-05,0.35840,0.37434,0.35771,0.36502,4.696269e+07,ADA,10.0,3.0,7.0,29.0,52.0,51.0,1.7,2.95,14.77,14077.4,250.792,63.0,Greed
5,2018-05-06,0.36502,0.36641,0.33538,0.34654,6.392687e+07,ADA,8.0,3.0,8.0,32.0,53.0,56.0,1.7,2.95,14.77,14077.4,250.792,67.0,Greed
6,2018-05-07,0.34599,0.34740,0.29100,0.33046,6.406094e+07,ADA,8.0,4.0,9.0,31.0,55.0,58.0,1.7,2.95,14.75,14077.4,250.792,56.0,Greed
7,2018-05-08,0.33079,0.34559,0.31630,0.32213,4.510501e+07,ADA,8.0,3.0,9.0,31.0,55.0,56.0,1.7,2.97,14.71,14077.4,250.792,62.0,Greed
8,2018-05-09,0.32213,0.32547,0.29800,0.31884,6.063555e+07,ADA,8.0,3.0,9.0,30.0,51.0,58.0,1.7,3.00,13.42,14077.4,250.792,53.0,Neutral
9,2018-05-10,0.31884,0.32210,0.29100,0.29121,3.821052e+07,ADA,7.0,3.0,8.0,30.0,55.0,63.0,1.7,2.97,13.23,14077.4,250.792,63.0,Greed


In [37]:
df.to_csv('data/df_features.csv', index=False)